In [0]:
%sql
CREATE OR REPLACE TEMP VIEW runtime_parameters AS

SELECT
    (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events) AS max_medical_date,

    (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events) AS max_pharmacy_date,

    LAST_DAY(
        ADD_MONTHS(
            LEAST(
                (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events),
                (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events)
            ), -1
        )
    ) AS end_date,

    CURRENT_DATE() AS run_date;

    SELECT * FROM runtime_parameters;

In [0]:
%sql
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.elaprase_procedure_code_lookup AS
SELECT * FROM VALUES
('99601','CPT','Infusion Administration','Home infusion/specialty drug administration; per visit (up to 2 hours)'),
('99602','CPT','Infusion Administration','Home infusion; each additional hour'),
('96365','CPT','Infusion Administration','Intravenous infusion, initial; up to 1 hour'),
('96366','CPT','Infusion Administration','Intravenous infusion, each additional hour'),
('J1743','HCPCS','Drug (IVIG)','Injection, immune globulin (IVIG), 500 mg'),
('S9357','HCPCS','Home Infusion Therapy (Supplies/Drugs)','Home infusion therapy; antibiotic, per diem'),
('S9379','HCPCS','Home Infusion Therapy (Supplies/Drugs)','Home infusion therapy; other drug (not otherwise classified), per diem'),
('38206','CPT','Stem Cell / Apheresis','Hematopoietic progenitor cell collection; apheresis'),
('38230','CPT','Bone Marrow Procedure','Bone marrow harvesting for transplantation'),
('38232','CPT','Bone Marrow Procedure','Bone marrow harvesting with preparation and storage'),
('38240','CPT','Stem Cell Transplant','Hematopoietic progenitor cell transplantation; allogeneic'),
('38241','CPT','Stem Cell Transplant','Hematopoietic progenitor cell transplantation; autologous'),
('38242','CPT','Stem Cell Transplant','Stem cell transplant with donor lymphocyte infusion'),
('38243','CPT','Stem Cell Transplant','Stem cell transplant; tandem transplant'),
('38250','CPT','Stem Cell Processing','Bone marrow or stem cell processing (e.g., cryopreservation)')
AS t(code, code_type, category, description);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_Patients AS

WITH base_claims AS (

    /* =========================
       MEDICAL CLAIMS
       ========================= */
    SELECT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI,
        BILLING_NPI AS BILLING_OR_PHARMACY_NPI,
        SERVICE_DATE AS CLAIM_DATE,
        MEDICAL_EVENT_ID AS CLAIM_ID,
        CAST(NULL AS STRING) AS CLAIM_STATUS,
        DIAGNOSIS_CODES,
        NDC11,
        PROCEDURE_CODE,
        KH_PLAN_ID AS KH_PLAN,
        PLACE_OF_SERVICE AS PLACE_OF_SERVICE_CODE,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE SERVICE_DATE BETWEEN '2023-01-01' AND (SELECT end_date FROM runtime_parameters)

    UNION ALL

    /* =========================
       PHARMACY CLAIMS
       ========================= */
    SELECT
        PATIENT_ID,
        PRESCRIBER_NPI AS HCP_NPI,
        PHARMACY_NPI AS BILLING_OR_PHARMACY_NPI,
        FILL_DATE AS CLAIM_DATE,
        PHARMACY_EVENT_ID AS CLAIM_ID,
        TRANSACTION_RESULT AS CLAIM_STATUS,
        DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
        NDC11,
        CAST(NULL AS STRING) AS PROCEDURE_CODE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        'PHARMACY' AS PLACE_OF_SERVICE_CODE,
        'PHARMACY_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE FILL_DATE BETWEEN '2023-01-01' AND (SELECT end_date FROM runtime_parameters)
),

/* =========================
   FILTERED CLAIMS
   ========================= */
filtered_claims AS (
    SELECT *
    FROM base_claims
    WHERE 
        DIAGNOSIS_CODES LIKE '%E761%' 
        OR DIAGNOSIS_CODES LIKE '%E763%'
        OR NDC11 IN ('54092070001','540920700')
        OR PROCEDURE_CODE IN (
            '99601','99602','96365','96366','J1743','S9357','S9379',
            '38206','38230','38232','38240','38241','38242','38243','38250'
        )
),

/* =========================
   TAGGING
   ========================= */
tagged_claims AS (

    SELECT
        f.*,

        CASE 
            WHEN DIAGNOSIS_CODES LIKE '%E761%' THEN 'SPECIFIED'
            WHEN DIAGNOSIS_CODES LIKE '%E763%' THEN 'UNSPECIFIED'
            ELSE NULL
        END AS DX_TYPE,

        CASE 
            WHEN NDC11 IN ('54092070001','540920700')
              OR PROCEDURE_CODE IN (
                  '99601','99602','96365','96366','J1743','S9357','S9379',
                  '38206','38230','38232','38240','38241','38242','38243','38250'
              )
            THEN 1 ELSE 0 
        END AS TREATMENT_FLAG,

        CASE 
            WHEN NDC11 IN ('54092070001','540920700')
              OR PROCEDURE_CODE = 'J1743'
            THEN 1 ELSE 0 
        END AS ELAPRASE_FLAG

    FROM filtered_claims f
),

/* =========================
   JOIN WITH LOOKUP
   ========================= */
claims_with_lookup AS (

    SELECT
        t.*,
        l.code AS procedure_code_lookup,
        l.description AS procedure_description,
        l.category AS procedure_category
    FROM tagged_claims t
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.elaprase_procedure_code_lookup l
        ON t.procedure_code = l.code
),

/* =========================
   PATIENT SUMMARY
   ========================= */
patient_summary AS (

    SELECT
        PATIENT_ID,
        COUNT(DISTINCT CASE WHEN DX_TYPE = 'SPECIFIED' THEN CLAIM_DATE END) AS spec_dx_cnt,
        COUNT(DISTINCT CASE WHEN DX_TYPE = 'UNSPECIFIED' THEN CLAIM_DATE END) AS unspec_dx_cnt,
        MAX(TREATMENT_FLAG) AS has_treatment,
        MAX(ELAPRASE_FLAG) AS has_elaprase
    FROM claims_with_lookup
    GROUP BY PATIENT_ID
),

eligible_patients AS (

    SELECT PATIENT_ID
    FROM patient_summary
    WHERE 
        (spec_dx_cnt >= 2 AND has_treatment = 1)
        OR
        (unspec_dx_cnt >= 2 AND has_elaprase = 1)
)

/* =========================
   FINAL OUTPUT
   ========================= */
SELECT DISTINCT
    CLAIM_ID,
    CLAIM_STATUS,
    CLAIM_DATE,
    PATIENT_ID,
    HCP_NPI,
    BILLING_OR_PHARMACY_NPI,
    PLACE_OF_SERVICE_CODE,

    /* original unified code */
    CASE 
        WHEN DX_TYPE IS NOT NULL THEN DIAGNOSIS_CODES
        ELSE COALESCE(NDC11, PROCEDURE_CODE)
    END AS CODE,

    /* NEW FIELDS */
    PROCEDURE_CODE,
    PROCEDURE_DESCRIPTION,
    PROCEDURE_CATEGORY,
    NDC11,

    TABLE_NAME

FROM claims_with_lookup
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
AND TREATMENT_FLAG = 1;

In [0]:
%sql
select count(*), count(distinct claim_id), count(distinct patient_id), count(distinct hcp_npi) from MPSII_Patients

In [0]:
%sql
-- CREATE OR REPLACE TEMP VIEW MPSII_Patients AS
-- WITH
-- /* ============================================================================
--    1) ELIGIBILITY COHORT BUILD
--    ========================================================================== */

-- MPSII_Diagnoses_Specified AS (
--     SELECT DISTINCT
--         PATIENT_ID,
--         COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI,
--         BILLING_NPI AS BILLING_OR_PHARMACY_NPI,
--         SERVICE_DATE AS CLAIM_DATE,
--         MEDICAL_EVENT_ID AS CLAIM_ID,
--         CAST(NULL AS STRING) AS CLAIM_STATUS,
--         DIAGNOSIS_CODES,
--         KH_PLAN_ID AS KH_PLAN,
--         PLACE_OF_SERVICE AS PLACE_OF_SERVICE_CODE,
--         'MEDICAL_EVENTS' AS TABLE_NAME
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E761%'
--       AND SERVICE_DATE BETWEEN '2023-01-01' AND (SELECT end_date FROM runtime_parameters)

--     UNION

--     SELECT DISTINCT
--         PATIENT_ID,
--         PRESCRIBER_NPI AS HCP_NPI,
--         PHARMACY_NPI AS BILLING_OR_PHARMACY_NPI,
--         FILL_DATE AS CLAIM_DATE,
--         PHARMACY_EVENT_ID AS CLAIM_ID,
--         TRANSACTION_RESULT AS CLAIM_STATUS,
--         DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
--         COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
--         'PHARMACY' AS PLACE_OF_SERVICE_CODE,
--         'PHARMACY_EVENTS' AS TABLE_NAME
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E761'
--       AND FILL_DATE BETWEEN '2023-01-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- Patients_2Dx_Specified AS (
--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Specified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT CLAIM_DATE) >= 2
-- ),

-- MPSII_Diagnoses_Unspecified AS (
--     SELECT DISTINCT
--         PATIENT_ID,
--         COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI,
--         BILLING_NPI AS BILLING_OR_PHARMACY_NPI,
--         SERVICE_DATE AS CLAIM_DATE,
--         MEDICAL_EVENT_ID AS CLAIM_ID,
--         CAST(NULL AS STRING) AS CLAIM_STATUS,
--         DIAGNOSIS_CODES,
--         KH_PLAN_ID AS KH_PLAN,
--         PLACE_OF_SERVICE AS PLACE_OF_SERVICE_CODE,
--         'MEDICAL_EVENTS' AS TABLE_NAME
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E763%'
--       AND SERVICE_DATE BETWEEN '2023-01-01' AND (SELECT end_date FROM runtime_parameters)

--     UNION

--     SELECT DISTINCT
--         PATIENT_ID,
--         PRESCRIBER_NPI AS HCP_NPI,
--         PHARMACY_NPI AS BILLING_OR_PHARMACY_NPI,
--         FILL_DATE AS CLAIM_DATE,
--         PHARMACY_EVENT_ID AS CLAIM_ID,
--         TRANSACTION_RESULT AS CLAIM_STATUS,
--         DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
--         COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
--         'PHARMACY' AS PLACE_OF_SERVICE_CODE,
--         'PHARMACY_EVENTS' AS TABLE_NAME
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E763'
--       AND FILL_DATE BETWEEN '2023-01-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- Patients_2Dx_Unspecified AS (
--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Unspecified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT CLAIM_DATE) >= 2
-- ),

-- /* ============================================================================
--    2) TREATMENT EVIDENCE
--    ========================================================================== */

-- MPSII_Treatment_Universe AS (

--     SELECT DISTINCT
--         PATIENT_ID,
--         COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI,
--         BILLING_NPI AS BILLING_OR_PHARMACY_NPI,
--         NDC11 AS CODE,
--         MEDICAL_EVENT_ID AS CLAIM_ID,
--         SERVICE_DATE AS CLAIM_DATE,
--         CAST(NULL AS STRING) AS CLAIM_STATUS,
--         PLACE_OF_SERVICE AS PLACE_OF_SERVICE_CODE,
--         KH_PLAN_ID AS KH_PLAN,
--         'MEDICAL_EVENTS' AS TABLE_NAME
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE NDC11 IN ('54092070001', '540920700')
--       AND SERVICE_DATE BETWEEN '2023-01-01' AND (SELECT end_date FROM runtime_parameters)

--     UNION ALL

--     SELECT DISTINCT
--         PATIENT_ID,
--         PRESCRIBER_NPI AS HCP_NPI,
--         PHARMACY_NPI AS BILLING_OR_PHARMACY_NPI,
--         NDC11 AS CODE,
--         PHARMACY_EVENT_ID AS CLAIM_ID,
--         FILL_DATE AS CLAIM_DATE,
--         TRANSACTION_RESULT AS CLAIM_STATUS,
--         'PHARMACY' AS PLACE_OF_SERVICE_CODE,
--         COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
--         'PHARMACY_EVENTS' AS TABLE_NAME
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE NDC11 IN ('54092070001', '540920700')
--       AND FILL_DATE BETWEEN '2023-01-01' AND (SELECT end_date FROM runtime_parameters)

--     UNION ALL

--     SELECT DISTINCT
--         PATIENT_ID,
--         RENDERING_NPI AS HCP_NPI,
--         BILLING_NPI AS BILLING_OR_PHARMACY_NPI,
--         PROCEDURE_CODE AS CODE,
--         MEDICAL_EVENT_ID AS CLAIM_ID,
--         SERVICE_DATE AS CLAIM_DATE,
--         CAST(NULL AS STRING) AS CLAIM_STATUS,
--         PLACE_OF_SERVICE AS PLACE_OF_SERVICE_CODE,
--         KH_PLAN_ID AS KH_PLAN,
--         'MEDICAL_EVENTS' AS TABLE_NAME
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE PROCEDURE_CODE IN (
--         '99601','99602','96365','96366','J1743','S9357','S9379',
--         '38206','38230','38232','38240','38241','38242','38243','38250'
--     )
--       AND SERVICE_DATE BETWEEN '2023-01-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- MPSII_Treatment_Elaprase_Only AS (
--     SELECT DISTINCT PATIENT_ID
--     FROM (
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('54092070001', '540920700')
--           AND SERVICE_DATE BETWEEN '2023-01-01' AND (SELECT end_date FROM runtime_parameters)

--         UNION ALL

--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('54092070001', '540920700')
--           AND FILL_DATE BETWEEN '2023-01-01' AND (SELECT end_date FROM runtime_parameters)

--         UNION ALL

--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE = 'J1743'
--           AND SERVICE_DATE BETWEEN '2023-01-01' AND (SELECT end_date FROM runtime_parameters)
--     ) t
-- ),

-- /* ============================================================================
--    3) COHORT LOGIC
--    ========================================================================== */

-- Patients_2Dx_Specified_With_Treatment AS (
--     SELECT DISTINCT
--         p.PATIENT_ID
--     FROM Patients_2Dx_Specified p
--     INNER JOIN (
--         SELECT DISTINCT PATIENT_ID
--         FROM MPSII_Treatment_Universe
--     ) t
--         ON p.PATIENT_ID = t.PATIENT_ID
-- ),

-- Patients_Incremental_Unspecified AS (
--     SELECT DISTINCT
--         p.PATIENT_ID
--     FROM Patients_2Dx_Unspecified p
--     INNER JOIN MPSII_Treatment_Elaprase_Only t
--         ON p.PATIENT_ID = t.PATIENT_ID
--     WHERE p.PATIENT_ID NOT IN (
--         SELECT PATIENT_ID
--         FROM Patients_2Dx_Specified_With_Treatment
--     )
-- ),

-- Eligible_Patients AS (
--     SELECT PATIENT_ID
--     FROM Patients_2Dx_Specified_With_Treatment

--     UNION

--     SELECT PATIENT_ID
--     FROM Patients_Incremental_Unspecified
-- ),

-- /* ============================================================================
--    4) DETAIL ROWS TO RETURN
--    ========================================================================== */

-- MPSII_Patient_Detail AS (
--     SELECT DISTINCT
--         PATIENT_ID,
--         HCP_NPI,
--         BILLING_OR_PHARMACY_NPI,
--         CLAIM_DATE,
--         CLAIM_ID,
--         CLAIM_STATUS,
--         DIAGNOSIS_CODES AS CODE,
--         KH_PLAN,
--         PLACE_OF_SERVICE_CODE,
--         TABLE_NAME
--     FROM MPSII_Diagnoses_Specified

--     UNION

--     SELECT DISTINCT
--         PATIENT_ID,
--         HCP_NPI,
--         BILLING_OR_PHARMACY_NPI,
--         CLAIM_DATE,
--         CLAIM_ID,
--         CLAIM_STATUS,
--         DIAGNOSIS_CODES AS CODE,
--         KH_PLAN,
--         PLACE_OF_SERVICE_CODE,
--         TABLE_NAME
--     FROM MPSII_Diagnoses_Unspecified

--     UNION

--     SELECT DISTINCT
--         PATIENT_ID,
--         HCP_NPI,
--         BILLING_OR_PHARMACY_NPI,
--         CLAIM_DATE,
--         CLAIM_ID,
--         CLAIM_STATUS,
--         CODE,
--         KH_PLAN,
--         PLACE_OF_SERVICE_CODE,
--         TABLE_NAME
--     FROM MPSII_Treatment_Universe
-- )

-- /* ============================================================================
--    5) FINAL OUTPUT
--    ========================================================================== */

-- SELECT DISTINCT
--     d.CLAIM_ID,
--     d.CLAIM_STATUS,
--     d.CLAIM_DATE,
--     d.PATIENT_ID,
--     d.HCP_NPI,
--     d.BILLING_OR_PHARMACY_NPI,
--     d.PLACE_OF_SERVICE_CODE,
--     d.CODE,
--     d.TABLE_NAME
-- FROM MPSII_Patient_Detail d
-- INNER JOIN Eligible_Patients e
--     ON d.PATIENT_ID = e.PATIENT_ID
-- ;

In [0]:
%sql
CREATE OR REPLACE TABLE cmpa_insights_internal_schema.MPSII_Patients_Enriched AS
SELECT DISTINCT
    mp.CLAIM_ID,
    mp.CLAIM_STATUS,
    mp.CLAIM_DATE,
    mp.PATIENT_ID,

    /* =========================
       PATIENT DEMO
       ========================= */
    YEAR(CURRENT_DATE) - YEAR(pd.PATIENT_YOB) AS PATIENT_AGE,

    CASE 
        WHEN YEAR(CURRENT_DATE) - YEAR(pd.PATIENT_YOB) < 17 THEN 'PEDIATRIC'
        ELSE 'ADULT'
    END AS AGE_GROUP,

    pg.PATIENT_STATE,
    pg.PATIENT_ZIP,

    /* =========================
       CLAIM INFO
       ========================= */
    mp.PLACE_OF_SERVICE_CODE,
    CASE
        WHEN mp.PLACE_OF_SERVICE_CODE = 'PHARMACY' THEN 'PHARMACY'
        ELSE pos.DESCRIPTION
    END AS PLACE_OF_SERVICE_DESCRIPTION,

    mp.TABLE_NAME,

    /* =========================
       PROCEDURE / DRUG ENRICHMENT
       ========================= */
    mp.PROCEDURE_CODE,
    mp.PROCEDURE_DESCRIPTION,
    mp.PROCEDURE_CATEGORY,
    mp.NDC11,

    /* =========================
       MPSII FLAGS (NEW)
       ========================= */

    /* Drug flag */
    CASE 
        WHEN mp.NDC11 IN ('54092070001') THEN 1 
        ELSE 0 
    END AS HAS_MPSII_DRUG,

    /* Procedure flag */
    CASE 
        WHEN mp.PROCEDURE_CODE IN (
            '99601','99602','96365','96366','J1743',
            'S9357','S9379',
            '38206','38230','38232','38240','38241','38242','38243','38250'
        ) THEN 1 
        ELSE 0 
    END AS HAS_MPSII_PROCEDURE,

    /* Combined flag (VERY useful for filtering) */
    CASE 
        WHEN mp.NDC11 IN ('54092070001')
          OR mp.PROCEDURE_CODE IN (
            '99601','99602','96365','96366','J1743',
            'S9357','S9379',
            '38206','38230','38232','38240','38241','38242','38243','38250'
          )
        THEN 1 ELSE 0
    END AS HAS_MPSII_TREATMENT,

    /* =========================
       CODE TYPE
       ========================= */
    CASE 
        WHEN HAS_MPSII_DRUG = 1 AND HAS_MPSII_PROCEDURE = 1 THEN 'BOTH'
        WHEN HAS_MPSII_DRUG = 1 THEN 'MPSII_DRUG'
        WHEN HAS_MPSII_PROCEDURE = 1 THEN 'MPSII_PROCEDURE'
        ELSE 'OTHER'
    END AS CODE_TYPE,

    /* =========================
       THERAPY CLASS
       ========================= */
    CASE 
        WHEN mp.NDC11 IN ('54092070001','540920700') 
             OR mp.PROCEDURE_CODE = 'J1743'
        THEN 'ELAPRASE'
        WHEN mp.PROCEDURE_CATEGORY LIKE '%Stem Cell%' 
        THEN 'Stem Cell Transplantation'
        WHEN mp.PROCEDURE_CATEGORY LIKE '%Infusion%' 
        THEN 'INFUSION_SUPPORT'
        ELSE 'OTHER'
    END AS THERAPY_CLASS,

    /* =========================
       HCP DETAILS
       ========================= */
    mp.HCP_NPI,
    TRIM(CONCAT(COALESCE(pr.FIRST_NAME, ''), ' ', COALESCE(pr.LAST_NAME, ''))) AS HCP_NAME,
    pr.PROVIDER_ADDRESS AS HCP_ADDRESS,
    hcp_ztm.CITY AS HCP_CITY,
    hcp_ztm.STATE AS HCP_STATE,
    pr.PROVIDER_ZIP AS HCP_ZIP,

    /* =========================
       HCO DETAILS
       ========================= */
    hco.ORGANIZATION_NAME AS HCO_NAME,
    hco.PROVIDER_ADDRESS AS HCO_ADDRESS,
    hco_ztm.CITY AS HCO_CITY,
    hco_ztm.STATE AS HCO_STATE,
    hco.PROVIDER_ZIP AS HCO_ZIP,
    /* =========================
   MPSII TREATMENT TYPE (NEW - CLEAN LOGIC)
   ========================= */
CASE 
    WHEN mp.NDC11 = '54092070001'
         OR mp.PROCEDURE_CODE = 'J1743'
    THEN 'ELAPRASE'

    WHEN mp.PROCEDURE_CODE IN (
        '99601','99602','96365','96366',
        'S9357','S9379',
        '38206','38230','38232',
        '38240','38241','38242','38243','38250'
    )
    THEN 'OTHER MPSII PROC'

    ELSE 'NON MPSII'
END AS MPSII_TREATMENT_TYPE

FROM MPSII_Patients mp

/* =========================
   JOINS 
   ========================= */
LEFT JOIN com_edp_prd.com_raw.kom_patient_demographics pd
    ON mp.PATIENT_ID = pd.PATIENT_ID

LEFT JOIN (
    SELECT
        PATIENT_ID,
        MAX(PATIENT_STATE) AS PATIENT_STATE,
        MAX(PATIENT_ZIP) AS PATIENT_ZIP
    FROM com_edp_prd.com_raw.kom_patient_geography
    GROUP BY PATIENT_ID
) pg
    ON mp.PATIENT_ID = pg.PATIENT_ID

LEFT JOIN (
    SELECT
        NPI,
        MAX(FIRST_NAME) AS FIRST_NAME,
        MAX(LAST_NAME) AS LAST_NAME,
        MAX(PROVIDER_ADDRESS) AS PROVIDER_ADDRESS,
        MAX(PROVIDER_ZIP) AS PROVIDER_ZIP
    FROM com_edp_prd.com_raw.kom_providers
    GROUP BY NPI
) pr
    ON mp.HCP_NPI = pr.NPI

LEFT JOIN (
    SELECT
        NPI,
        MAX(ORGANIZATION_NAME) AS ORGANIZATION_NAME,
        MAX(PROVIDER_ADDRESS) AS PROVIDER_ADDRESS,
        MAX(PROVIDER_ZIP) AS PROVIDER_ZIP
    FROM com_edp_prd.com_raw.kom_providers
    WHERE PROVIDER_TYPE = 'ORGANIZATION'
    GROUP BY NPI
) hco
    ON mp.BILLING_OR_PHARMACY_NPI = hco.NPI

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.pos_description pos
    ON TRY_CAST(mp.PLACE_OF_SERVICE_CODE AS INT) = TRY_CAST(pos.CODE AS INT)

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping hcp_ztm
    ON pr.PROVIDER_ZIP = hcp_ztm.ZIPCODE

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping hco_ztm
    ON hco.PROVIDER_ZIP = hco_ztm.ZIPCODE
;

In [0]:
%sql
select count(*), count(distinct claim_id), count(distinct patient_id), count(distinct hcp_npi) from cmpa_insights_internal_schema.MPSII_Patients_enriched

In [0]:
%sql
select * from cmpa_insights_internal_schema.MPSII_Patients_Enriched 
-- where PLACE_OF_SERVICE_DESCRIPTION is null

In [0]:
%sql
SELECT *
FROM (
    SELECT
        PLACE_OF_SERVICE_DESCRIPTION,
        PROCEDURE_CODE,
        CLAIM_ID
    FROM cmpa_insights_internal_schema.MPSII_Patients_Enriched
    WHERE PROCEDURE_CODE IS NOT NULL
)
PIVOT (
    COUNT(CLAIM_ID)
    FOR PROCEDURE_CODE IN (
        '99601','99602','96365','96366','J1743',
        'S9357','S9379','38206','38230','38232',
        '38240','38241','38242','38243','38250'
    )
)
ORDER BY PLACE_OF_SERVICE_DESCRIPTION;


In [0]:
%python

# =============================
# Step 0: Imports
# =============================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =============================
# Step 1: Load data from Spark
# =============================
df = spark.table("cmpa_insights_internal_schema.MPSII_Patients_Enriched") \
    .select("PLACE_OF_SERVICE_DESCRIPTION", "PROCEDURE_CATEGORY", "CLAIM_ID") \
    .dropna() \
    .toPandas()

print("Data loaded:", df.shape)
display(df.head())

# =============================
# Step 2: Create Pivot Table
# =============================
pivot = pd.pivot_table(
    df,
    index="PLACE_OF_SERVICE_DESCRIPTION",
    columns="PROCEDURE_CATEGORY",
    values="CLAIM_ID",
    aggfunc="count",
    fill_value=0
)

# =============================
# Step 3: Add Totals
# =============================
pivot["Total"] = pivot.sum(axis=1)
pivot.loc["Total"] = pivot.sum()

print("\nPivot Table:")
display(pivot)

# =============================
# Step 4: Heatmap (Best View)
# =============================
plt.figure(figsize=(14,6))

sns.heatmap(
    pivot,
    annot=True,
    fmt=".0f",
    linewidths=0.5
)

plt.title("Claims by Place of Service vs Procedure Category")
plt.xlabel("Procedure Category")
plt.ylabel("Place of Service")

plt.tight_layout()
plt.show()

# =============================
# Step 5: Stacked Bar (Cleaner for Business)
# =============================

# Remove total row/column for chart
pivot_chart = pivot.drop("Total", axis=0).drop("Total", axis=1)

pivot_chart.plot(
    kind="bar",
    stacked=True,
    figsize=(12,6)
)

plt.title("Procedure Mix by Place of Service")
plt.xlabel("Place of Service")
plt.ylabel("Claims")

plt.legend(title="Procedure Category", bbox_to_anchor=(1.05, 1))
plt.tight_layout()

plt.show()

# =============================
# Step 6: Top Categories Only (Optional - Cleaner)
# =============================
top_cols = pivot_chart.sum().sort_values(ascending=False).head(5).index

pivot_top = pivot_chart[top_cols]

pivot_top.plot(
    kind="bar",
    stacked=True,
    figsize=(12,6)
)

plt.title("Top 5 Procedure Categories by Place of Service")
plt.xlabel("Place of Service")
plt.ylabel("Claims")

plt.legend(title="Procedure Category", bbox_to_anchor=(1.05, 1))
plt.tight_layout()

plt.show()

In [0]:
%sql
SELECT
    COUNT(DISTINCT PATIENT_ID) AS total_patients,
    COUNT(DISTINCT CASE WHEN AGE_GROUP = 'PEDIATRIC' THEN PATIENT_ID END) AS pediatric_patients_lt17,
    COUNT(DISTINCT CASE WHEN AGE_GROUP = 'ADULT' THEN PATIENT_ID END) AS adult_patients
FROM cmpa_insights_internal_schema.MPSII_Patients_Enriched ;

In [0]:
%sql
SELECT
    PATIENT_STATE,
    COUNT(DISTINCT PATIENT_ID) AS patient_count
FROM cmpa_insights_internal_schema.MPSII_Patients_Enriched
GROUP BY PATIENT_STATE
ORDER BY patient_count DESC;

In [0]:
%sql
SELECT
    THERAPY_CLASS,

    COUNT(DISTINCT PATIENT_ID) AS total_patients,

    COUNT(DISTINCT CASE 
        WHEN AGE_GROUP = 'PEDIATRIC' THEN PATIENT_ID 
    END) AS pediatric_patients,

    COUNT(DISTINCT CASE 
        WHEN AGE_GROUP = 'ADULT' THEN PATIENT_ID 
    END) AS adult_patients

FROM cmpa_insights_internal_schema.MPSII_Patients_Enriched
GROUP BY THERAPY_CLASS
ORDER BY total_patients DESC;

In [0]:
%sql
SELECT
    THERAPY_CLASS,
    COUNT(DISTINCT PATIENT_ID) AS pediatric_patients
FROM cmpa_insights_internal_schema.MPSII_Patients_Enriched
WHERE AGE_GROUP = 'PEDIATRIC'
GROUP BY THERAPY_CLASS;

In [0]:
%sql
SELECT
    HCO_NAME,
    COUNT(DISTINCT PATIENT_ID) AS total_patients,
    COUNT(DISTINCT CASE WHEN THERAPY_CLASS = 'ELAPRASE' THEN PATIENT_ID END) AS elaprase_patients
FROM cmpa_insights_internal_schema.MPSII_Patients_Enriched
GROUP BY HCO_NAME
ORDER BY total_patients DESC;

In [0]:
%sql
SELECT
    PLACE_OF_SERVICE_DESCRIPTION,
    COUNT(DISTINCT PATIENT_ID) AS patient_count
FROM cmpa_insights_internal_schema.MPSII_Patients_Enriched
GROUP BY PLACE_OF_SERVICE_DESCRIPTION
ORDER BY patient_count DESC;

In [0]:
%sql
SELECT
    HCP_NAME,
    COUNT(DISTINCT PATIENT_ID) AS patient_count
FROM cmpa_insights_internal_schema.MPSII_Patients_Enriched
GROUP BY HCP_NAME
ORDER BY patient_count DESC
LIMIT 20;

In [0]:
%sql
SELECT
    DATE_TRUNC('MONTH', CLAIM_DATE) AS month,
    COUNT(DISTINCT PATIENT_ID) AS patients
FROM cmpa_insights_internal_schema.MPSII_Patients_Enriched
GROUP BY month
ORDER BY month;

In [0]:
%sql
SELECT
    CODE_TYPE,
    COUNT(DISTINCT PATIENT_ID) AS patients
FROM cmpa_insights_internal_schema.MPSII_Patients_Enriched
GROUP BY CODE_TYPE;

# ELAPRASE ONLY 

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_Elaprase_Only_Patients AS

WITH base_claims AS (

    /* =========================
       MEDICAL CLAIMS
       ========================= */
    SELECT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI,
        BILLING_NPI AS BILLING_OR_PHARMACY_NPI,
        SERVICE_DATE AS CLAIM_DATE,
        MEDICAL_EVENT_ID AS CLAIM_ID,
        CAST(NULL AS STRING) AS CLAIM_STATUS,
        DIAGNOSIS_CODES,
        NDC11,
        PROCEDURE_CODE,
        KH_PLAN_ID AS KH_PLAN,
        PLACE_OF_SERVICE AS PLACE_OF_SERVICE_CODE,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE SERVICE_DATE BETWEEN '2023-01-01' AND (SELECT end_date FROM runtime_parameters)

    UNION ALL

    /* =========================
       PHARMACY CLAIMS
       ========================= */
    SELECT
        PATIENT_ID,
        PRESCRIBER_NPI AS HCP_NPI,
        PHARMACY_NPI AS BILLING_OR_PHARMACY_NPI,
        FILL_DATE AS CLAIM_DATE,
        PHARMACY_EVENT_ID AS CLAIM_ID,
        TRANSACTION_RESULT AS CLAIM_STATUS,
        DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
        NDC11,
        CAST(NULL AS STRING) AS PROCEDURE_CODE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        'PHARMACY' AS PLACE_OF_SERVICE_CODE,
        'PHARMACY_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE FILL_DATE BETWEEN '2023-01-01' AND (SELECT end_date FROM runtime_parameters)
),

/* =========================
   KEEP ONLY RELEVANT CLAIMS
   ========================= */
filtered_claims AS (

    SELECT *
    FROM base_claims
    WHERE 
        DIAGNOSIS_CODES LIKE '%E761%' 
        OR DIAGNOSIS_CODES LIKE '%E763%'
        OR NDC11 IN ('54092070001','540920700')
        OR PROCEDURE_CODE = 'J1743'
),

/* =========================
   TAGGING LAYER
   ========================= */
tagged_claims AS (

    SELECT
        *,
        
        CASE 
            WHEN DIAGNOSIS_CODES LIKE '%E761%' THEN 'SPECIFIED'
            WHEN DIAGNOSIS_CODES LIKE '%E763%' THEN 'UNSPECIFIED'
            ELSE NULL
        END AS DX_TYPE,

        CASE 
            WHEN NDC11 IN ('54092070001','540920700')
              OR PROCEDURE_CODE = 'J1743'
            THEN 1 ELSE 0 
        END AS TREATMENT_FLAG,

        CASE 
            WHEN NDC11 IN ('54092070001','540920700')
              OR PROCEDURE_CODE = 'J1743'
            THEN 1 ELSE 0 
        END AS ELAPRASE_FLAG

    FROM filtered_claims
),

/* =========================
   PATIENT LEVEL SUMMARY
   ========================= */
patient_summary AS (

    SELECT
        PATIENT_ID,

        COUNT(DISTINCT CASE WHEN DX_TYPE = 'SPECIFIED' THEN CLAIM_DATE END) AS spec_dx_cnt,
        COUNT(DISTINCT CASE WHEN DX_TYPE = 'UNSPECIFIED' THEN CLAIM_DATE END) AS unspec_dx_cnt,

        MAX(TREATMENT_FLAG) AS has_treatment,
        MAX(ELAPRASE_FLAG) AS has_elaprase

    FROM tagged_claims
    GROUP BY PATIENT_ID
),

/* =========================
   ELIGIBLE COHORT
   ========================= */
eligible_patients AS (

    SELECT PATIENT_ID
    FROM patient_summary
    WHERE 
        (spec_dx_cnt >= 2 AND has_treatment = 1)
        OR
        (unspec_dx_cnt >= 2 AND has_elaprase = 1)
)

/* =========================
   FINAL OUTPUT
   ========================= */
SELECT DISTINCT
    CLAIM_ID,
    CLAIM_STATUS,
    CLAIM_DATE,
    PATIENT_ID,
    HCP_NPI,
    BILLING_OR_PHARMACY_NPI,
    PLACE_OF_SERVICE_CODE,
    CASE 
        WHEN DX_TYPE IS NOT NULL THEN DIAGNOSIS_CODES
        ELSE COALESCE(NDC11, PROCEDURE_CODE)
    END AS CODE,
    TABLE_NAME
FROM tagged_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
AND TREATMENT_FLAG = 1
;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_Elaprase_Only_Patients_Enriched AS
SELECT DISTINCT
    mp.CLAIM_ID,
    mp.CLAIM_STATUS,
    mp.CLAIM_DATE,
    mp.PATIENT_ID,
    YEAR(CURRENT_DATE) - YEAR(pd.PATIENT_YOB) AS PATIENT_AGE,
    pg.PATIENT_STATE,
    pg.PATIENT_ZIP,
    mp.PLACE_OF_SERVICE_CODE,
    CASE
        WHEN mp.PLACE_OF_SERVICE_CODE = 'PHARMACY' THEN 'PHARMACY'
        ELSE pos.DESCRIPTION
    END AS PLACE_OF_SERVICE_DESCRIPTION,
    mp.HCP_NPI,
    TRIM(CONCAT(COALESCE(pr.FIRST_NAME, ''), ' ', COALESCE(pr.LAST_NAME, ''))) AS HCP_NAME,

    pr.PROVIDER_ADDRESS AS HCP_ADDRESS,
    hcp_ztm.CITY AS HCP_CITY,
    hcp_ztm.STATE AS HCP_STATE,
    pr.PROVIDER_ZIP AS HCP_ZIP,

    hco.ORGANIZATION_NAME AS HCO_NAME,
    hco.PROVIDER_ADDRESS AS HCO_ADDRESS,
    hco_ztm.CITY AS HCO_CITY,
    hco_ztm.STATE AS HCO_STATE,
    hco.PROVIDER_ZIP AS HCO_ZIP,

    mp.TABLE_NAME

FROM MPSII_Elaprase_Only_Patients mp
LEFT JOIN com_edp_prd.com_raw.kom_patient_demographics pd
    ON mp.PATIENT_ID = pd.PATIENT_ID

LEFT JOIN (
    SELECT
        PATIENT_ID,
        MAX(PATIENT_STATE) AS PATIENT_STATE,
        MAX(PATIENT_ZIP) AS PATIENT_ZIP
    FROM com_edp_prd.com_raw.kom_patient_geography
    GROUP BY PATIENT_ID
) pg
    ON mp.PATIENT_ID = pg.PATIENT_ID

LEFT JOIN (
    SELECT
        NPI,
        MAX(FIRST_NAME) AS FIRST_NAME,
        MAX(LAST_NAME) AS LAST_NAME,
        MAX(PROVIDER_ADDRESS) AS PROVIDER_ADDRESS,
        MAX(PROVIDER_ZIP) AS PROVIDER_ZIP
    FROM com_edp_prd.com_raw.kom_providers
    GROUP BY NPI
) pr
    ON mp.HCP_NPI = pr.NPI

LEFT JOIN (
    SELECT
        NPI,
        MAX(ORGANIZATION_NAME) AS ORGANIZATION_NAME,
        MAX(PROVIDER_ADDRESS) AS PROVIDER_ADDRESS,
        MAX(PROVIDER_ZIP) AS PROVIDER_ZIP
    FROM com_edp_prd.com_raw.kom_providers
    WHERE PROVIDER_TYPE = 'ORGANIZATION'
    GROUP BY NPI
) hco
    ON mp.BILLING_OR_PHARMACY_NPI = hco.NPI

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.pos_description pos
    ON TRY_CAST(mp.PLACE_OF_SERVICE_CODE AS INT) = TRY_CAST(pos.CODE AS INT)

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping hcp_ztm
    ON pr.PROVIDER_ZIP = hcp_ztm.ZIPCODE

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping hco_ztm
    ON hco.PROVIDER_ZIP = hco_ztm.ZIPCODE
;

In [0]:
%sql
select count(*), count(distinct claim_id), count(distinct patient_id), count(distinct hcp_npi) from MPSII_Elaprase_Only_Patients_Enriched

In [0]:
%sql
select * from MPSII_Elaprase_Only_Patients_Enriched